In [0]:
%sql
-- Are there emerging districts with accelerating growth?
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_accelerating_districts AS
WITH yearly_district_permit AS (
    SELECT 
    dd.cd,
    sd.year AS permit_year,
    COUNT(fp.permit_nbr) AS total_permits
    FROM la_lakehouse.gold.fact_permits AS fp
    INNER JOIN la_lakehouse.gold.dim_date AS sd
    ON fp.submitted_date_key = sd.date_key
    LEFT JOIN la_lakehouse.gold.dim_district AS dd
    ON fp.district_key = dd.district_key
    WHERE dd.cd IS NOT NULL
    GROUP BY dd.cd, sd.year
),
year_over_year_calulation AS (
    SELECT 
    cd,
    permit_year,
    total_permits,
    LAG(total_permits,1) OVER(PARTITION BY cd ORDER BY permit_year) AS previous_year_total,
    ROUND(100.0 * (total_permits - LAG(total_permits,1) OVER(PARTITION BY cd ORDER BY permit_year))
        / NULLIF(LAG(total_permits,1) OVER(PARTITION BY cd ORDER BY permit_year),0),2)
    AS yoy_growth_pct
    FROM yearly_district_permit AS ydp
)

SELECT 
cd,
permit_year,
total_permits,
previous_year_total,
yoy_growth_pct,
ROUND(yoy_growth_pct - LAG(yoy_growth_pct,1) OVER(PARTITION BY cd ORDER BY permit_year),2) AS growth_acceleration_pts
FROM year_over_year_calulation
ORDER BY cd, permit_year ASC;



In [0]:
%sql
-- Are there community plan areas with accelerating growth?
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_accelerating_cpa AS
WITH yearly_cpa_permit AS (
    SELECT 
    cpa.cpa,
    sd.year AS permit_year,
    COUNT(fp.permit_nbr) AS total_permits
    FROM la_lakehouse.gold.fact_permits AS fp
    INNER JOIN la_lakehouse.gold.dim_date AS sd
    ON fp.submitted_date_key = sd.date_key
    LEFT JOIN la_lakehouse.gold.dim_community_plan_area AS cpa
    ON fp.cpa_key = cpa.cpa_key
    GROUP BY cpa.cpa, sd.year
),
year_over_year_calulation AS (
    SELECT 
    cpa,
    permit_year,
    total_permits,
    LAG(total_permits,1) OVER(PARTITION BY cpa ORDER BY permit_year) AS previous_year_total,
    ROUND(100.0 * (total_permits - LAG(total_permits,1) OVER(PARTITION BY cpa ORDER BY permit_year))
        / NULLIF(LAG(total_permits,1) OVER(PARTITION BY cpa ORDER BY permit_year),0),2)
    AS yoy_growth_pct
    FROM yearly_cpa_permit AS ydp
)

SELECT 
cpa,
permit_year,
total_permits,
previous_year_total,
yoy_growth_pct,
ROUND(yoy_growth_pct - LAG(yoy_growth_pct,1) OVER(PARTITION BY cpa ORDER BY permit_year),2) AS growth_acceleration_cpa_pts
FROM year_over_year_calulation
ORDER BY cpa, permit_year ASC;



In [0]:
%sql
--  How concentrated is activity — does a small number of districts dominate total volume?
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_concentrated_districts AS
WITH district_market_shares AS (
    SELECT
    dd.cd,
    COUNT(*) AS total_permits,
    ROUND(100.0 * COUNT(*)/ SUM(COUNT(*)) OVER(),2) AS pct_of_total
    FROM la_lakehouse.gold.fact_permits AS fp
    LEFT JOIN la_lakehouse.gold.dim_district AS dd
    ON fp.district_key = dd.district_key
    WHERE dd.cd IS NOT NULL
    GROUP BY dd.cd
),
cumulative_percentage AS (
    SELECT
    cd,
    total_permits,
    pct_of_total,
    DENSE_RANK() OVER(ORDER BY total_permits DESC) AS district_rank,
    ROUND(SUM(pct_of_total) OVER(ORDER BY total_permits DESC, cd ASC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),2) AS cumulative_pct
    FROM district_market_shares
)
SELECT 
cd,
total_permits,
pct_of_total,
cumulative_pct,
CASE WHEN district_rank <=3 THEN TRUE ELSE FALSE END AS is_top_3,
CASE WHEN district_rank <=5 THEN TRUE ELSE FALSE END AS is_top_5,
CASE WHEN (cumulative_pct - pct_of_total) < 80.0 THEN TRUE ELSE FALSE END AS is_in_pareto_80
FROM cumulative_percentage
ORDER BY district_rank ASC

#Testing Views

In [0]:
%sql
SELECT * 
FROM vw_accelerating_districts;

In [0]:
%sql
SELECT *
FROM la_lakehouse.gold.vw_concentrated_districts;

In [0]:
%sql
SELECT *
FROM la_lakehouse.gold.vw_community_impact_submission_rate;